# <center>Clustering Analysis<center>

<p>Team Name:
<p>Student Names:

## Instructions
Use generic coding style unless hard-coded values are really necessary.<br>
Your code must be efficient and use self-explanatory naming.<br>
Use appropriate Python library methods for each task instead of using loops.<br>
Run your entire code and save. Then submit this <b>saved</b> copy.

## Imports

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures

## Read Data

## Visual Exploration of Data

### Histograms

### Distributions

### Box-Whisker Plots

### Violin Plots

## Data Quality & Cleaning

Instruction: Add a comment for each method

In [ ]:
# Check for missing values
missing_counts = df_norm.isnull().sum()
if missing_counts.sum() > 0:
    missing_pct = (missing_counts / len(df_norm)) * 100
    missing_summary = pd.DataFrame({
        'Column': missing_counts.index,
        'Missing Count': missing_counts.values,
        'Percentage': missing_pct.values
    })
    missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
    print("Missing values found:\n", missing_summary)
else:
    print("No missing values detected.")

# Check for duplicate records
duplicate_rows = df_norm.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows}")

# Check for duplicate fish_ids
duplicate_ids = df_norm.duplicated(subset=['fish_id']).sum()
print(f"Duplicate fish_id values: {duplicate_ids}")

# Outlier detection using IQR
print("\nOutlier Detection (IQR Method):")
for feature in feature_cols:
    Q1 = df_norm[feature].quantile(0.25)
    Q3 = df_norm[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((df_norm[feature] < lower_bound) | (df_norm[feature] > upper_bound)).sum()
    print(f"{feature}: {outliers} outliers outside [{lower_bound:.4f}, {upper_bound:.4f}]")

# Validate normalized feature ranges
print("\nData Range Validation:")
for feature in feature_cols:
    min_val = df_norm[feature].min()
    max_val = df_norm[feature].max()
    in_range = ((df_norm[feature] >= 0) & (df_norm[feature] <= 1)).all()
    print(f"{feature}: [{min_val:.6f}, {max_val:.6f}], in range [0,1]: {in_range}")

# Check data types
print("\nData Types:")
print(df_norm.dtypes)

# Summary statistics
print("\nSummary Statistics:")
print(df_norm[feature_cols].describe())

# Class distribution
print("\nClass Distribution:")
print(df_norm['fish_class'].value_counts())

# Cleaning operations
print("\nCleaning Operations:")
initial_rows = len(df_norm)
df_cleaned = df_norm.drop_duplicates(subset=['fish_id'], keep='first')
rows_removed = initial_rows - len(df_cleaned)
print(f"Removed {rows_removed} duplicate fish_id records")

# Clip normalized features to [0, 1] if needed
for feature in feature_cols:
    out_of_range = ((df_cleaned[feature] < 0) | (df_cleaned[feature] > 1)).sum()
    if out_of_range > 0:
        df_cleaned[feature] = df_cleaned[feature].clip(0, 1)
        print(f"Clipped {out_of_range} {feature} values to [0, 1]")

df_norm = df_cleaned.copy()
print(f"\nFinal dataset shape: {df_norm.shape}")

## Handling Redundancy

### X-square Test

### Correlation Analysis

### Visual Exploration (scatter-plot matrix)

## Dimensionality Reduction

### PCA

## Discretization

### Histogram of Discretized Attribute

### X-square Test of Discretized Attributes

### Visual Exploration (scatter-plot matrix) of Discretized Attributes

## Feature Selection/Generation

### Select Features

In [ ]:
# Encode fish_class for sklearn methods
le = LabelEncoder()
y_encoded = le.fit_transform(df_norm['fish_class'])

# Feature selection without PCA
print("=== FEATURE SELECTION WITHOUT PCA ===")
X_original = df_norm[feature_cols].values

# ANOVA F-test feature selection
selector_f = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X_original, y_encoded)
f_scores = pd.DataFrame({
    'Feature': feature_cols,
    'F-Score': selector_f.scores_,
    'P-Value': selector_f.pvalues_
}).sort_values('F-Score', ascending=False)
print('ANOVA F-test scores:')
print(f_scores)

# Select top features based on F-scores
top_k = 3
selector_f_top = SelectKBest(score_func=f_classif, k=top_k)
X_selected_f = selector_f_top.fit_transform(X_original, y_encoded)
selected_features_f = [feature_cols[i] for i in selector_f_top.get_support(indices=True)]
print(f'\nTop {top_k} features selected (F-test): {selected_features_f}')

# Mutual information feature selection
mi_scores = mutual_info_classif(X_original, y_encoded, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_cols,
    'MI-Score': mi_scores
}).sort_values('MI-Score', ascending=False)
print('\nMutual Information scores:')
print(mi_df)

selector_mi = SelectKBest(score_func=mutual_info_classif, k=top_k)
X_selected_mi = selector_mi.fit_transform(X_original, y_encoded)
selected_features_mi = [feature_cols[i] for i in selector_mi.get_support(indices=True)]
print(f'\nTop {top_k} features selected (MI): {selected_features_mi}')

# Create dataframe with selected features
df_selected_no_pca = df_norm[['fish_id', 'fish_class'] + selected_features_f].copy()
print(f'\nDataFrame shape after selection (no PCA): {df_selected_no_pca.shape}')

# Feature selection with PCA
print("\n=== FEATURE SELECTION WITH PCA ===")
X_pca = df_norm_pca[feature_cols_pca].values

# ANOVA F-test on PCA components
selector_f_pca = SelectKBest(score_func=f_classif, k='all')
selector_f_pca.fit(X_pca, y_encoded)
f_scores_pca = pd.DataFrame({
    'Feature': feature_cols_pca,
    'F-Score': selector_f_pca.scores_,
    'P-Value': selector_f_pca.pvalues_
}).sort_values('F-Score', ascending=False)
print('ANOVA F-test scores on PCA components:')
print(f_scores_pca)

# Mutual information on PCA components
mi_scores_pca = mutual_info_classif(X_pca, y_encoded, random_state=42)
mi_df_pca = pd.DataFrame({
    'Feature': feature_cols_pca,
    'MI-Score': mi_scores_pca
}).sort_values('MI-Score', ascending=False)
print('\nMutual Information scores on PCA components:')
print(mi_df_pca)

# Use all PCA components
df_selected_with_pca = df_norm_pca[['fish_id', 'fish_class'] + feature_cols_pca].copy()
print(f'\nDataFrame shape after selection (with PCA): {df_selected_with_pca.shape}')

# Summary
print("\n=== SUMMARY ===")
print(f"Without PCA: {len(selected_features_f)} features selected: {selected_features_f}")
print(f"With PCA: {len(feature_cols_pca)} components: {feature_cols_pca}")

### Generate Features

In [ ]:
# Feature generation without PCA
print("=== FEATURE GENERATION WITHOUT PCA ===")
df_gen_no_pca = df_norm[['fish_id', 'fish_class'] + selected_features_f].copy()

# Create ratio features
if 'avg_intensity' in selected_features_f and 'avg_gradient' in selected_features_f:
    df_gen_no_pca['ratio_intensity_gradient'] = (
        df_gen_no_pca['avg_intensity'] / (df_gen_no_pca['avg_gradient'] + 1e-10)
    )
    print("Created ratio feature: ratio_intensity_gradient")

# Create interaction features
for i, feat1 in enumerate(selected_features_f):
    for feat2 in selected_features_f[i+1:]:
        interaction_name = f'interaction_{feat1}_{feat2}'.replace('avg_', '').replace('std_', '')
        df_gen_no_pca[interaction_name] = df_gen_no_pca[feat1] * df_gen_no_pca[feat2]
        print(f"Created interaction feature: {interaction_name}")

# Create statistical aggregation features
if len(selected_features_f) > 1:
    df_gen_no_pca['mean_features'] = df_gen_no_pca[selected_features_f].mean(axis=1)
    df_gen_no_pca['std_features'] = df_gen_no_pca[selected_features_f].std(axis=1)
    print("Created statistical features: mean_features, std_features")

print(f"\nTotal features (without PCA): {len(df_gen_no_pca.columns) - 2}")
print(f"Generated features: {[col for col in df_gen_no_pca.columns if col not in ['fish_id', 'fish_class'] + selected_features_f]}")

# Feature generation with PCA
print("\n=== FEATURE GENERATION WITH PCA ===")
df_gen_with_pca = df_norm_pca[['fish_id', 'fish_class'] + feature_cols_pca].copy()

# Create squared PCA features
df_gen_with_pca['PC1_squared'] = df_gen_with_pca['PC1'] ** 2
df_gen_with_pca['PC2_squared'] = df_gen_with_pca['PC2'] ** 2
print("Created squared features: PC1_squared, PC2_squared")

# Create interaction between PCA components
df_gen_with_pca['PC1_PC2'] = df_gen_with_pca['PC1'] * df_gen_with_pca['PC2']
print("Created interaction feature: PC1_PC2")

# Create magnitude feature
df_gen_with_pca['PC_magnitude'] = np.sqrt(df_gen_with_pca['PC1']**2 + df_gen_with_pca['PC2']**2)
print("Created magnitude feature: PC_magnitude")

# Create angle feature
df_gen_with_pca['PC_angle'] = np.arctan2(df_gen_with_pca['PC2'], df_gen_with_pca['PC1'])
print("Created angle feature: PC_angle")

print(f"\nTotal features (with PCA): {len(df_gen_with_pca.columns) - 2}")
print(f"Generated features: {[col for col in df_gen_with_pca.columns if col not in ['fish_id', 'fish_class'] + feature_cols_pca]}")

# Generate Clusters

## K-means

## Hierarchical

# Evaluation of Clusters

See instructions provided in the report template

## <center> REFERENCES </center>
List resources (book, internet page, etc.) that you used to complete this challenge.
- https://scikit-learn.org/stable/modules/feature_selection.html
- https://scikit-learn.org/stable/modules/preprocessing.html